# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ku-ro-wa/flyrank-ml-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = 1/30 or 1/90 of the data for any individual page that will be used to predict its performance for the next 30 days.

In [3]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN {HF_TOKEN})")
rel = "hf://datasets/FlyRank/internship-warehouse"

# Check schema
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet') LIMIT 0").show()

# Basic count
con.sql(f"""
    SELECT COUNT(*) AS n_rows
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""").show()

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────┐
│  n_rows  │
│  int64   │
├──────────┤
│ 78835655 │
└──────────┘



In [16]:
# A single row
con.sql(f"""
    SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_pageviews,
    ga4_sessions,
    scroll_events
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE report_date = DATE '2026-03-15'
LIMIT 1;""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬─────────────────────────┬──────────────────────────┬─────────────────┬────────────┬──────────────────┬───────────────┬──────────────┬───────────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ gsc_impressions │ gsc_clicks │ gsc_avg_position │ ga4_pageviews │ ga4_sessions │ scroll_events │
│    date     │         varchar         │         varchar          │      int64      │   int64    │      double      │     int64     │    int64     │     int64     │
├─────────────┼─────────────────────────┼──────────────────────────┼─────────────────┼────────────┼──────────────────┼───────────────┼──────────────┼───────────────┤
│ 2026-03-15  │ client_a60a11451483af1c │ content_a7c85d1e50d132ef │               0 │          0 │             NULL │             0 │            0 │             0 │
└─────────────┴─────────────────────────┴──────────────────────────┴─────────────────┴────────────┴──────────────────┴───────────────┴──────────────┴───────────────┘



In [17]:
# 30 rows = 30 days worth of data for one page, could also be 90 rows by changing the integer after LIMIT but would basically look the same as below.
con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_pageviews,
        ga4_sessions,
        scroll_events
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE client_hash_id = 'client_a60a11451483af1c' AND content_hash_id = 'content_a7c85d1e50d132ef'
    ORDER BY report_date ASC
    LIMIT 30;""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬─────────────────────────┬──────────────────────────┬─────────────────┬────────────┬──────────────────┬───────────────┬──────────────┬───────────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ gsc_impressions │ gsc_clicks │ gsc_avg_position │ ga4_pageviews │ ga4_sessions │ scroll_events │
│    date     │         varchar         │         varchar          │      int64      │   int64    │      double      │     int64     │    int64     │     int64     │
├─────────────┼─────────────────────────┼──────────────────────────┼─────────────────┼────────────┼──────────────────┼───────────────┼──────────────┼───────────────┤
│ 2025-11-05  │ client_a60a11451483af1c │ content_a7c85d1e50d132ef │               0 │          0 │             NULL │          NULL │         NULL │          NULL │
│ 2025-11-06  │ client_a60a11451483af1c │ content_a7c85d1e50d132ef │               0 │          0 │             NULL │          NULL │         NULL │          NULL │
│ 20

In [11]:
con.sql(f"""
    SELECT MIN(report_date) AS min_report_date, MAX(report_date) AS max_report_date
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet');""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┬─────────────────┐
│ min_report_date │ max_report_date │
│      date       │      date       │
├─────────────────┼─────────────────┤
│ 2025-01-27      │ 2026-06-30      │
└─────────────────┴─────────────────┘



## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Fields are listed by category (Just for fact_content_daily_performance):

Feature:
- client_has_gsc
- client_has_ga4
- gsc_data_available
- ga4_data_available
- gsc_avg_position
- ga4_users
- ga4_engaged_sessions
- ga4_total_engagement_sec
- sessions_organic
- sessions_direct
- sessions_referral
- sessions_social
- sessions_paid
- sessions_ai
- ai_chatgpt (and all other ai_insert_other_models_here)
- scroll_events


Label:
- gsc_impressions (Can be used as a feature when used as a rolling value)
- gsc_clicks (Can be used as a feature when used as a rolling value)
- ga4_pageviews (Can be used as a feature when used as a rolling value)
- ga4_sessions (Can be used as a feature when used as a rolling value)

Context:
- report_date
- client_hash_id
- content_hash_id

Excluded:
- gsc_sum_position (Can be derived from gsc_avg_position * gsc_impressions. Excluding keeps the model simpler)

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [19]:
sql = """
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_avg_position,
    gsc_impressions,
    gsc_sum_position,
    -- Re‑compute the sum from avg × impressions
    (gsc_avg_position * gsc_impressions) AS derived_sum,
    -- Boolean flag that is true when they match
    (gsc_sum_position = (gsc_avg_position * gsc_impressions)) AS matches
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE gsc_avg_position IS NOT NULL
  AND gsc_impressions IS NOT NULL
LIMIT 20;      -- LIMIT so it doesn't run on the full table
""".format(rel=rel)

df = con.sql(sql).df()
print(df)

sql2 = """
SELECT
    COUNT(*)                               AS total_rows,
    SUM(CASE WHEN gsc_sum_position = (gsc_avg_position * gsc_impressions) THEN 1 ELSE 0 END) AS matching_rows,
    SUM(CASE WHEN gsc_sum_position <> (gsc_avg_position * gsc_impressions) THEN 1 ELSE 0 END) AS mismatched_rows
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE gsc_avg_position IS NOT NULL
  AND gsc_impressions IS NOT NULL;
  """.format(rel=rel)

df2 = con.sql(sql2).df()
print(df2)

   report_date           client_hash_id           content_hash_id  \
0   2025-01-27  client_9958f0a7ae1df715  content_3b70a18ea133b2bb   
1   2025-01-27  client_9958f0a7ae1df715  content_fe8e8155ce1d47a2   
2   2025-01-27  client_9958f0a7ae1df715  content_b4462a1b90640058   
3   2025-01-27  client_9958f0a7ae1df715  content_c899aef92518c714   
4   2025-01-27  client_9958f0a7ae1df715  content_c7c1d2e68d9d0964   
5   2025-01-27  client_9958f0a7ae1df715  content_c782fa8abd4fce5e   
6   2025-01-27  client_9958f0a7ae1df715  content_ae5e5fd6edff550f   
7   2025-01-27  client_9958f0a7ae1df715  content_a64143f6e4a21ffe   
8   2025-01-27  client_9958f0a7ae1df715  content_e281674658070602   
9   2025-01-27  client_9958f0a7ae1df715  content_658f53fa439c66ca   
10  2025-01-27  client_9958f0a7ae1df715  content_da9cd3207814ec8d   
11  2025-01-27  client_9958f0a7ae1df715  content_96fe7476fada560c   
12  2025-01-27  client_9958f0a7ae1df715  content_89c6c2e17e412e20   
13  2025-01-27  client_9958f0a7ae1

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  matching_rows  mismatched_rows
0    28970001     27911241.0        1058760.0


Okay there are mismatched rows but as you can see with the first query matched values still may be flagged as False, potentially due to some rounding issues or whatever.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

- Why a page's performance is what it is: You can see correlations, but you cannot infer causality.
- Exact user‑level behavior: All metrics are aggregated at the client_hash_id + content_hash_id + date grain. There is no session‑level, device‑level, geographic, or demographic breakdown  
- Specific search queries: GSC give totals (impressions, clicks, average position) but not the actual queries made
- Content details: Any specifics about the actual content
- Any results beyond session metrics: Tangible business impact or ROI can't be derived without any further conversion or downstream events
- Not all rows have GSC and/or GA4 data
- Daily activity means that you can't track hourly trends or changes'
- Cross‑platform attribution: Sessions are split into organic, direct, referral, social, paid, and ai, but there is no attribution model that ties a user’s path across channels
- External context or events going on globally that may affect the statistics


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.